# Saving a figure

**Saving a figure -- for a slide, a paper, or a web page.**

**What it shows:**

- figsize is in INCHES, dpi is dots per inch; together they set the pixels
- text does not scale with the figure, so a shrunk figure has tiny labels
- PNG for slides and the web, SVG/PDF for print -- and why
- bbox_inches="tight" to stop matplotlib cropping your labels off

The mistake this prevents: drawing a chart, shrinking it into a slide, and wondering why nobody in the back row can read the axis.

---

*Chapter:* `foundations` — matplotlib's actual mechanics  
*Run the cells in order.* Every figure is also written to `viz/output/foundations/`, which is what the Streamlit gallery (`viz/project/gallery.py`) reads.


## Setup

These lines are how every notebook in the folder finds `vizkit.py`, which holds the save helpers and the seeded sample data. The data is seeded on purpose: your figures should come out identical to everyone else's.

`save()` writes each figure into `viz/output/` **and** leaves it on screen here. The trailing `;` on those calls only stops the notebook echoing the path it returns.


In [ ]:
%matplotlib inline

# A notebook has no __file__, so find viz/ by walking up from this
# notebook's own folder until vizkit.py turns up.
import sys
from pathlib import Path

VIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
           if (p / "vizkit.py").exists())
sys.path.insert(0, str(VIZ))

import matplotlib.pyplot as plt
import numpy as np

from vizkit import save, OUTPUT

# Where save() files this lesson's output: viz/output/foundations/
LESSON = "foundations/saving"


## One chart, drawn over and over

A helper that draws the same little chart every time, so the only thing changing below is the size and the format.


In [ ]:
x = np.linspace(0, 10, 100)


def demo_plot(ax, title):
    ax.plot(x, np.sin(x), label="signal")
    ax.set(xlabel="time (s)", ylabel="amplitude", title=title)
    ax.legend()


## 1. Size the figure for its destination

`figsize` is in **inches**; `dpi` is dots per inch. Multiply them and you get pixels. Pick the inches from where the chart is going, and let dpi decide how crisp it is.


In [ ]:
# Do NOT draw big and shrink later: the text shrinks with it.
targets = {
    "slide": (10, 5.6),        # 16:9, read from across a room
    "paper-column": (3.5, 2.6),  # a single journal column
    "web": (8, 4.5),
}

for name, size in targets.items():
    fig, ax = plt.subplots(figsize=size)
    demo_plot(ax, f"{name}: figsize={size}")
    save(fig, LESSON, name)


## 2. What shrinking does to text

The two panels come out the same width on screen. The left one was drawn small; the right one was drawn at 16×10 and shrunk. Font sizes are in points, and points do not shrink with the figure — which is why the right one is unreadable.


In [ ]:
# Both panels below end up the same width on this page. The left one was drawn
# small; the right one was drawn large and then scaled down. Same code, very
# different readability.
fig, ax = plt.subplots(figsize=(4, 2.5))
demo_plot(ax, "drawn at 4x2.5 inches")
save(fig, LESSON, "drawn-small", dpi=160)

fig, ax = plt.subplots(figsize=(16, 10))
demo_plot(ax, "drawn at 16x10, then scaled down")
save(fig, LESSON, "drawn-large", dpi=40);     # low dpi = same pixel width


## 3. Raster vs vector

Same figure, three files. Look at the byte counts printed below: PNG stores pixels, SVG and PDF store the shapes, so vector formats stay sharp at any zoom and usually weigh less for line art.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
demo_plot(ax, "same figure, three formats")

folder = OUTPUT / "foundations"
folder.mkdir(parents=True, exist_ok=True)

sizes = {}
for ext in ("png", "svg", "pdf"):
    path = folder / f"saving-formats.{ext}"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    sizes[ext] = path.stat().st_size
plt.close(fig)

for ext, size in sizes.items():
    print(f"  saving-formats.{ext:<4} {size/1024:7.1f} KB")

print(f"""
  Size it for where it is going -- do not draw big and shrink:
    slide         figsize=(10, 5.6), dpi 150+
    paper column  figsize=(3.5, 2.6), and raise the font size
    web           figsize=(8, 4.5), dpi 100-150

  Format:
    PNG  pixels. Slides, web, anything with a photo or a heatmap.
    SVG  shapes. Sharp at any zoom, editable in Illustrator/Inkscape.
    PDF  shapes. What journals want.

  Always: bbox_inches="tight", or your y label ends up outside the image.
""")


## Try it yourself

Edit the cells above and re-run them — that is what the notebook is for.

1. Save the `paper-column` figure again with `plt.rcParams['font.size'] = 11`. Is it readable at 3.5 inches wide now?
2. Add a `hexbin` of 50,000 points to `demo_plot` and re-run section 3. Which format wins on size now, and why did the ranking flip?
3. Save one figure with `bbox_inches=None` and a y label of 'a very long axis label indeed'. Open the file — what got cut off?


In [ ]:
# your turn


---

**Previous:** [`foundations/subplots_grid`](subplots_grid.ipynb)  
**Next:** [`color/palettes`](../color/palettes.ipynb)
